# Giai đoạn 3: Business Insights từ OCPM

Đây là phần tạo ra **giá trị kinh doanh thực tế** — trả lời những câu hỏi mà Traditional Process Mining không thể:

| # | Câu hỏi | Góc nhìn OCPM |
|---|---------|---------------|
| Q1 | Số lượng Offer ảnh hưởng đến thời gian xử lý Application? | Application ↔ Offer interaction |
| Q2 | Bottleneck nằm ở tầng Application hay Offer? | Per-object waiting time |
| Q3 | Khi Offer bị từ chối → Application có bị ảnh hưởng? | Object trajectory analysis |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
# Tải dữ liệu đã xử lý
events_df    = pd.read_csv('output/bpi2017_events.csv')
objects_df   = pd.read_csv('output/bpi2017_objects.csv')
relations_df = pd.read_csv('output/bpi2017_relations.csv')

events_df['ocel:timestamp']    = pd.to_datetime(events_df['ocel:timestamp'], utc=True, errors='coerce')
relations_df['ocel:timestamp'] = pd.to_datetime(relations_df['ocel:timestamp'], utc=True, errors='coerce')

# Dữ liệu gốc để lấy thông tin Application
df_raw = pd.read_csv('../data/bpi-challenge-2017/bpi_2017_cleaned.csv')
df_raw['time:timestamp'] = pd.to_datetime(df_raw['time:timestamp'], utc=True, errors='coerce')

print("✅ Dữ liệu đã tải xong")

## Q1: Số lượng Offer ảnh hưởng đến thời gian xử lý Application như thế nào?

> **Hypothesis:** Ngân hàng phát hành càng nhiều Offer → Application mất càng nhiều thời gian xử lý

In [ ]:
# Tính thời gian xử lý của từng Application
app_duration = df_raw.groupby('case:concept:name').agg(
    start_time = ('time:timestamp', 'min'),
    end_time   = ('time:timestamp', 'max')
).reset_index()
app_duration['duration_days'] = (
    app_duration['end_time'] - app_duration['start_time']
).dt.total_seconds() / 86400

# Tính số Offer của từng Application
offer_objects = objects_df[objects_df['ocel:type'] == 'offer'].copy()
offer_objects['parent_application'] = None

# Lấy quan hệ Application → Offer từ relations
app_offer_rel = relations_df[relations_df['ocel:type'] == 'offer'][[
    'ocel:oid', 'ocel:activity']].drop_duplicates()

# Lấy số offer từ raw data
offer_per_app = (
    df_raw[df_raw['OfferID'].notna()]
    .groupby('case:concept:name')['OfferID']
    .nunique()
    .reset_index(name='num_offers')
)
offer_per_app['offer_group'] = offer_per_app['num_offers'].apply(
    lambda x: '1 Offer' if x == 1 else ('2 Offers' if x == 2 else '3+ Offers')
)

# Merge với duration
app_analysis = app_duration.merge(offer_per_app, on='case:concept:name', how='left')
app_analysis['offer_group'] = app_analysis['offer_group'].fillna('0 Offers')

print("📊 Phân phối theo số lượng Offer:")
display(app_analysis['offer_group'].value_counts().to_frame())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

order = ['0 Offers', '1 Offer', '2 Offers', '3+ Offers']
palette = {'0 Offers': '#95a5a6', '1 Offer': '#3498db',
           '2 Offers': '#f39c12', '3+ Offers': '#e74c3c'}

# Boxplot thời gian xử lý
data_filtered = app_analysis[app_analysis['duration_days'] < app_analysis['duration_days'].quantile(0.95)]
sns.boxplot(data=data_filtered, x='offer_group', y='duration_days',
            order=order, palette=palette, ax=axes[0], showfliers=False)
axes[0].set_title('Thời gian xử lý Application\ntheo số lượng Offer (OCPM Insight)', fontsize=13)
axes[0].set_xlabel('Số Offer phát hành', fontsize=11)
axes[0].set_ylabel('Thời gian xử lý (ngày)', fontsize=11)

# Thêm annotation trung bình
for i, grp in enumerate(order):
    med = data_filtered[data_filtered['offer_group'] == grp]['duration_days'].median()
    axes[0].text(i, med + 0.5, f'Median:\n{med:.1f}d', ha='center', fontsize=9,
                color='darkred', fontweight='bold')

# Bar chart số lượng
group_means = app_analysis.groupby('offer_group')['duration_days'].agg(
    mean='mean', count='count').reindex(order)
bars = axes[1].bar(order, group_means['mean'],
                   color=[palette[g] for g in order], edgecolor='white', alpha=0.85)
axes[1].set_title('Thời gian xử lý TRUNG BÌNH\ntheo số lượng Offer', fontsize=13)
axes[1].set_xlabel('Số Offer phát hành', fontsize=11)
axes[1].set_ylabel('Trung bình thời gian (ngày)', fontsize=11)
for bar, (grp, row) in zip(bars, group_means.iterrows()):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.2,
                f"{row['mean']:.1f}d\n(n={int(row['count']):,})",
                ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('output/q1_offer_count_vs_duration.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Đã lưu: output/q1_offer_count_vs_duration.png")

## Q2: Bottleneck nằm ở tầng Application hay Offer?

> **Hypothesis:** Sự chậm trễ 7 ngày không phân bổ đều — nó tập trung tại một điểm chuyển tiếp cụ thể giữa Application và Offer.

In [ ]:
# Tính waiting time giữa các sự kiện TRONG cùng một đối tượng
# Đây là cái OCPM làm được mà Traditional PM không thể!

# ── Waiting time trong Application lifecycle ──────────────────────────────────
app_events = relations_df[relations_df['ocel:type'] == 'application'].copy()
app_events = app_events.sort_values(['ocel:oid', 'ocel:timestamp'])
app_events['prev_timestamp'] = app_events.groupby('ocel:oid')['ocel:timestamp'].shift(1)
app_events['waiting_hours'] = (
    app_events['ocel:timestamp'] - app_events['prev_timestamp']
).dt.total_seconds() / 3600
app_events_clean = app_events.dropna(subset=['waiting_hours'])
app_events_clean = app_events_clean[app_events_clean['waiting_hours'] < app_events_clean['waiting_hours'].quantile(0.95)]

# ── Waiting time trong Offer lifecycle ───────────────────────────────────────
offer_events = relations_df[relations_df['ocel:type'] == 'offer'].copy()
offer_events = offer_events.sort_values(['ocel:oid', 'ocel:timestamp'])
offer_events['prev_timestamp'] = offer_events.groupby('ocel:oid')['ocel:timestamp'].shift(1)
offer_events['waiting_hours'] = (
    offer_events['ocel:timestamp'] - offer_events['prev_timestamp']
).dt.total_seconds() / 3600
offer_events_clean = offer_events.dropna(subset=['waiting_hours'])
offer_events_clean = offer_events_clean[offer_events_clean['waiting_hours'] < offer_events_clean['waiting_hours'].quantile(0.95)]

print("📊 Bottleneck Analysis — Waiting Time Trung bình:")
print(f"  • Tầng Application: {app_events_clean['waiting_hours'].mean():.2f} giờ / transition")
print(f"  • Tầng Offer:       {offer_events_clean['waiting_hours'].mean():.2f} giờ / transition")

In [ ]:
# Top bottleneck transitions theo từng perspective
app_transitions = app_events_clean.groupby('ocel:activity')['waiting_hours'].agg(
    mean='mean', median='median', count='count'
).sort_values('mean', ascending=False).head(8)

offer_transitions = offer_events_clean.groupby('ocel:activity')['waiting_hours'].agg(
    mean='mean', median='median', count='count'
).sort_values('mean', ascending=False).head(8)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

def plot_bottleneck(ax, df, title, color):
    y_pos = range(len(df))
    bars = ax.barh(y_pos, df['mean'].values, color=color, alpha=0.8, edgecolor='white')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df.index, fontsize=10)
    ax.invert_yaxis()
    ax.set_xlabel('Waiting time trung bình (giờ)', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    for i, (bar, (_, row)) in enumerate(zip(bars, df.iterrows())):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                f"{row['mean']:.1f}h\n(n={int(row['count']):,})",
                va='center', fontsize=8)
    ax.grid(axis='x', alpha=0.3)

plot_bottleneck(axes[0], app_transitions,
                '🔵 Bottleneck Tầng Application\n(Waiting time trước mỗi activity)', '#3498db')
plot_bottleneck(axes[1], offer_transitions,
                '🔴 Bottleneck Tầng Offer\n(Waiting time trước mỗi activity)', '#e74c3c')

plt.tight_layout()
plt.savefig('output/q2_bottleneck_by_layer.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Đã lưu: output/q2_bottleneck_by_layer.png")

## Q3: Khi Offer bị từ chối → Application bị ảnh hưởng như thế nào?

> **Hypothesis:** Application có Offer bị từ chối/hủy sẽ mất nhiều thời gian hơn và có tỷ lệ chấp nhận thấp hơn.

In [ ]:
# Phân loại Application theo kết cục của các Offers
# Lấy kết cục cuối cùng của từng Offer
offer_outcomes = offer_events.sort_values(['ocel:oid', 'ocel:timestamp']).groupby('ocel:oid').last()[['ocel:activity']].reset_index()
offer_outcomes.columns = ['OfferID', 'final_status']

# Phân loại kết cục
def classify_outcome(status):
    if 'Accept' in str(status): return 'Accepted'
    elif 'Refus' in str(status): return 'Refused'
    elif 'Cancel' in str(status): return 'Cancelled'
    else: return 'Pending'

offer_outcomes['outcome_class'] = offer_outcomes['final_status'].apply(classify_outcome)

# Merge với Application ID
offer_with_app = df_raw[df_raw['OfferID'].notna()][[
    'OfferID', 'case:concept:name']].drop_duplicates()
offer_outcomes = offer_outcomes.merge(offer_with_app, on='OfferID', how='left')

# Phân loại Application theo kết cục tệ nhất của Offer
def app_trajectory(group):
    statuses = set(group['outcome_class'])
    if 'Accepted' in statuses and len(statuses) == 1: return 'Chỉ Accepted'
    elif 'Accepted' in statuses: return 'Có Refused/Cancelled, cuối cùng Accepted'
    elif 'Refused' in statuses or 'Cancelled' in statuses: return 'Bị từ chối hoàn toàn'
    else: return 'Đang xử lý'

app_trajectory_df = offer_outcomes.groupby('case:concept:name').apply(app_trajectory).reset_index()
app_trajectory_df.columns = ['case:concept:name', 'app_trajectory']

# Merge với duration
analysis_q3 = app_duration.merge(app_trajectory_df, on='case:concept:name', how='left')
analysis_q3['app_trajectory'] = analysis_q3['app_trajectory'].fillna('Không có Offer')

print("📊 Phân phối Application theo trajectory của Offer:")
display(analysis_q3['app_trajectory'].value_counts().to_frame())

In [ ]:
# Thêm tỷ lệ chấp nhận
accepted_cases = df_raw[df_raw['concept:name'] == 'A_Accepted']['case:concept:name'].unique()
analysis_q3['is_accepted'] = analysis_q3['case:concept:name'].isin(accepted_cases).astype(int)

trajectory_summary = analysis_q3.groupby('app_trajectory').agg(
    count          = ('case:concept:name', 'count'),
    mean_duration  = ('duration_days', 'mean'),
    median_duration= ('duration_days', 'median'),
    acceptance_rate= ('is_accepted', 'mean')
).reset_index()
trajectory_summary['acceptance_rate_pct'] = trajectory_summary['acceptance_rate'] * 100

print("\n📊 Tác động của Offer trajectory đến Application:")
display(trajectory_summary.round(2))

In [ ]:
order3 = ['Không có Offer', 'Chỉ Accepted', 'Có Refused/Cancelled, cuối cùng Accepted', 'Bị từ chối hoàn toàn']
order3 = [o for o in order3 if o in trajectory_summary['app_trajectory'].values]
ts = trajectory_summary.set_index('app_trajectory').reindex(order3)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors3 = ['#95a5a6', '#2ecc71', '#f39c12', '#e74c3c'][:len(order3)]

# Duration
bars1 = axes[0].bar(range(len(ts)), ts['mean_duration'].values,
                    color=colors3, edgecolor='white', alpha=0.85)
axes[0].set_xticks(range(len(ts)))
axes[0].set_xticklabels([o.replace(', ', ',\n') for o in order3], fontsize=9, rotation=15, ha='right')
axes[0].set_title('Thời gian xử lý trung bình\ntheo Offer Trajectory', fontsize=13)
axes[0].set_ylabel('Ngày')
for bar, val in zip(bars1, ts['mean_duration'].values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.2,
                f'{val:.1f}d', ha='center', fontsize=10, fontweight='bold')

# Acceptance rate
bars2 = axes[1].bar(range(len(ts)), ts['acceptance_rate_pct'].values,
                    color=colors3, edgecolor='white', alpha=0.85)
axes[1].set_xticks(range(len(ts)))
axes[1].set_xticklabels([o.replace(', ', ',\n') for o in order3], fontsize=9, rotation=15, ha='right')
axes[1].set_title('Tỷ lệ chấp nhận (Acceptance Rate)\ntheo Offer Trajectory', fontsize=13)
axes[1].set_ylabel('Tỷ lệ (%)')
axes[1].set_ylim(0, 105)
for bar, val in zip(bars2, ts['acceptance_rate_pct'].values):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('output/q3_offer_trajectory_impact.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Đã lưu: output/q3_offer_trajectory_impact.png")

## Tổng kết Insights từ OCPM

In [ ]:
print("=" * 65)
print("🏆 OCPM BUSINESS INSIGHTS — BPI 2017")
print("=" * 65)

# Q1
mean_by_group = app_analysis.groupby('offer_group')['duration_days'].mean()
m1 = mean_by_group.get('1 Offer', 0)
m3 = mean_by_group.get('3+ Offers', 0)
print(f"\n📌 Q1 — Tác động số lượng Offer đến thời gian xử lý:")
print(f"   • 1 Offer  → Trung bình {m1:.1f} ngày")
print(f"   • 3+ Offer → Trung bình {m3:.1f} ngày")
if m3 > m1:
    print(f"   ⚠️  Kết luận: Mỗi Offer thêm vào làm tăng thêm ~{(m3-m1)/2:.1f} ngày!")

# Q2
print(f"\n📌 Q2 — Bottleneck theo tầng đối tượng:")
print(f"   • Tầng Application: {app_events_clean['waiting_hours'].mean():.1f}h trung bình / bước")
print(f"   • Tầng Offer:       {offer_events_clean['waiting_hours'].mean():.1f}h trung bình / bước")
top_bottleneck = app_transitions.index[0]
print(f"   ⚠️  Bước chậm nhất: '{top_bottleneck}'")

# Q3
print(f"\n📌 Q3 — Tác động Offer bị từ chối:")
for _, row in trajectory_summary.iterrows():
    print(f"   • {row['app_trajectory'][:40]:<42}: {row['mean_duration']:.1f}d, Accept={row['acceptance_rate_pct']:.1f}%")

print(f"\n🎯 KHUYẾN NGHỊ CHO NGÂN HÀNG (Phía quy trình):")
print(f"   1. Giới hạn số lần tạo Offer: Chỉ tạo tối đa 2 Offer/Application")
print(f"   2. Tập trung cải thiện bước '{top_bottleneck}' — đây là nút thắt cổ chai chính")
print(f"   3. Khi Offer bị từ chối, cần quy trình phục hồi nhanh để giữ chân khách hàng")
print("="*65)